### --- Day 4: Giant Squid ---

Puzzle description redacted as-per Advent of Code guidelines

You may find the puzzle description at: https://adventofcode.com/2021/day/4

In [2]:
#!import ../Utils.ipynb

In [3]:
var inputLines = LoadPuzzleInput(2021, 4);
WriteLines(inputLines, maxCols: 40);

Loading puzzle file: Day4.txt
Total lines: 601
Max line length: 289

76,69,38,62,33,48,81,2,64,21,80,90,29,99

17 45 62 28 73
39 12  0 52  5
87 48 50 85 44


In [4]:
string[] testInputLines = 
[
    "7,4,9,5,11,17,23,2,0,14,21,24,10,16,13,6,15,25,12,22,18,20,8,19,3,26,1",
    "",
    "22 13 17 11  0",
    " 8  2 23  4 24",
    "21  9 14 16  7",
    " 6 10  3 18  5",
    " 1 12 20 15 19",
    "",
    " 3 15  0  2 22",
    " 9 18 13 17  5",
    "19  8  7 25 23",
    "20 11 10 24  4",
    "14 21 16 12  6",
    "",
    "14 21 17 24  4",
    "10 16 15  9 19",
    "18  8 23 26 20",
    "22 11 13  6  5",
    " 2  0 12  3  7",
];

Ok, for the first part of this one, I think we can break down each board into "slices", namely the rows / cols that can be matched in the bingo game. As the numbers are called, we find all slices that contain that number and remove it. The first slice to be emptied is the winner! We'll need to keep track of the board each slice belongs to, so we can compute the final score once the winning slice is found. 

In [5]:
using BingoSlice = (int boardId, SCG.HashSet<int> slice);

// Spoiler alert: for part 2 we need to know the sequence of winning boards, so
// we'll build the main function that does that, and just take the first result
// for part 1
int FindWinningBoard(string[] inputLines) => FindWinningBoards(inputLines).First();

IEnumerable<int> FindWinningBoards(string[] inputLines)
{
    var numberDraw = inputLines[0].ParseInts();
    var boardRows = inputLines[1..].SeparateBy(l => l is "");

    int[] makeCols(string s) => [..s.ParseInts()];
    int[][] makeBoard(string[] rows) => [..rows.Select(makeCols)];
    int[][][] boardLookup = [..boardRows.Select(makeBoard)];

    Dictionary<int, List<BingoSlice>> sliceLookup = new();
    foreach (var (boardId, board) in boardLookup.Index())
    {
        addBingoSlices(boardId, board);
    }

    HashSet<int> called = new();
    HashSet<int> boardsWon = new();
    foreach (var num in numberDraw)
    {
        called.Add(num);

        var foundMatches = sliceLookup.TryGetValue(num, out var matches);
        if (!foundMatches) { continue; }

        foreach (var (boardId, slice) in matches)
        {
            slice.Remove(num);
            if (slice.Count is 0)
            {
                var newWinner = boardsWon.Add(boardId);
                if (!newWinner) { continue; }
                
                yield return foundWinner(num, boardId);
            }
        }
    }

    int foundWinner(int num, int boardId)
    {
        var boardSet = boardLookup[boardId].SelectMany(row => row).ToHashSet();
        boardSet.ExceptWith(called);
        return boardSet.Sum() * num;
    }

    void addBingoSlices(int boardId, int[][] board)
    {
        var rows = board.Select(row => row.ToHashSet());
        var cols = Range(0, 5).Select(i => board.Select(row => row[i]).ToHashSet());
        var boardSlices = rows.Concat(cols);

        foreach (var slice in boardSlices)
        foreach (var num in slice)
        {
            sliceLookup[num] = sliceLookup.GetValueOrDefault(num, new());
            sliceLookup[num].Add((boardId, slice));
        }
    }
}

In [6]:
// The score of the winning board can now be calculated ... the final score, 188
// * 24 = 4512.

var testAnswer = FindWinningBoard(testInputLines);
Console.WriteLine(testAnswer);

4512


In [7]:
// To guarantee victory against the giant squid, figure out which board will win
// first. What will your final score be if you choose that board?

var part1Answer = FindWinningBoard(inputLines);
Console.WriteLine(part1Answer);

2496


In [8]:
// 2496 is correct!
Ensure(2496, part1Answer);

### --- Part Two ---

Puzzle description redacted as-per Advent of Code guidelines

You may find the puzzle description at: https://adventofcode.com/2021/day/4

Ok, for part 2, I think we can modify part 1 to return the list of boards as they win, then for part 1, we take the first board, and for part 2, we take the last.

In [10]:
int FindLastWinningBoard(string[] inputLines) => FindWinningBoards(inputLines).Last();

In [11]:
// In the above example ... the second board would have a sum of unmarked
// numbers equal to 148 for a final score of 148 * 13 = 1924.

var part2TestAnswer = FindLastWinningBoard(testInputLines);
Console.WriteLine(part2TestAnswer);

1924


In [12]:
// Figure out which board will win last. Once it wins, what would its final
// score be?

var part2Answer = FindLastWinningBoard(inputLines);
Console.WriteLine(part2Answer);

25925


In [13]:
// 25925 is correct!
Ensure(25925, part2Answer);